In [ ]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis =1)
df.head()

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

df_clean = df.copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Traffic_Level', 'Time_of_Day']) #Dropping the rows without the target vrariable and also critical features like traffic level and time of the day
print(f"After dropping: {df_clean.shape}")

df_clean['Weather'] = df_clean['Weather'].fillna("Unknown")
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())



In [ ]:
# Task 3: Write your code here:
print(f"Before: {df_clean.shape}")
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)
print(f"After: {df_clean.shape}")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler() #AFTER TRAIN TEST SPLITTING THE DATA
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
check_target_distribution(df_clean, "Delivery_Time") #NO TARGET IMBALANCE

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split, KFold
feature_cols = ['Distance_km',	'Weather', 	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min'	,'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
model = RandomForestRegressor()

mae = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)
n_splits = 5

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training model...")

    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae_score = mean_absolute_error(y_test, y_pred)
  mae.append(mae_score)

print("THE AVERAGE ERROR WE GOT IT - ")
print(np.mean(mae))


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
'feature': feature_cols,
'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

print("\n\n\n\nWE FIND THAT THE DISTANCE IS THE BIGGEST FACTOR AFFECTING DELIVERY TIME")

In [ ]:
# Task 2: Write your code here:
plt.hist(y_pred)

In [ ]:
# Task Bonus: Write your code here:]

mae2 = []
%pip install kagglehub catboost lightgbm tqdm -q
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae_score2 = mean_absolute_error(y_test, y_pred)
    mae2.append(mae_score)

print("Average Error")
print(np.mean(mae2))